# fhir4ds.dqm

Digital Quality Measure (DQM) evaluation and clinical audit. This notebook demonstrates how to evaluate a FHIR Measure and generate explainable narratives for every patient decision.

In [ ]:
# Install the unified package with measure evaluation support
%pip install "fhir4ds-v2[measures]"

In [ ]:
import json
import tempfile
from pathlib import Path
import fhir4ds
from fhir4ds.cql import FHIRDataLoader
from fhir4ds.dqm import MeasureEvaluator

con = fhir4ds.create_connection()
loader = FHIRDataLoader(con)

# Load sample patient and encounter data
resources = [
    {"resourceType": "Patient", "id": "p1", "active": True, "birthDate": "1965-04-20"},
    {"resourceType": "Encounter", "id": "e1", "status": "finished", "subject": {"reference": "Patient/p1"}}
]
for r in resources:
    loader.load_resource(r)

### Define the Measure
A measure consists of a CQL library (the logic) and a FHIR Measure resource (the metadata).

In [ ]:
cql_text = """
library BPControl version '1.0'
using FHIR version '4.0.1'
context Patient

define "Initial Population":
    exists ([Encounter] E where E.status = 'finished')
"""
cql_file = Path(tempfile.mktemp(suffix=".cql"))
cql_file.write_text(cql_text)

measure_json = {
    "resourceType": "Measure",
    "id": "test-measure",
    "group": [{
        "population": [
            {"code": {"coding": [{"code": "initial-population"}]}, "criteria": {"expression": "Initial Population"}}
        ]
    }]
}

### Evaluate with Audit Narratives
Enable `AuditMode.FULL` and `generate_narratives=True` to get explainable results.

In [ ]:
evaluator = MeasureEvaluator(conn=con)
result = evaluator.evaluate(
    measure_bundle=measure_json,
    cql_library_path=str(cql_file),
)

# Display the result
df = result.dataframe
df[["patient_id", "initial_population"]]